<a href="https://colab.research.google.com/github/devanshh019/codeexpo/blob/main/DATA_CREATION_FUSIONMODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import pickle

In [5]:



bg_df = pd.read_csv("/content/BACKROUND_processed.csv")
emotion_df = pd.read_pickle("/content/features.pkl")
text_df = pd.read_csv("/content/emergency_call_dataset_10000_realistic.csv")



In [6]:
bg_df["feature"] = bg_df["feature"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

In [7]:
emotion_df["features"][0].shape

(128, 200)

In [8]:
bg_groups = bg_df.groupby("class")
emo_groups = emotion_df.groupby("emotion_label")

In [9]:
fusion_rows = []

for _, row in text_df.iterrows():

    sound = row["sound_event"]
    emotion = row["emotion"]

    if sound in bg_groups.groups:
        bg_sample = bg_groups.get_group(sound).sample(1)["feature"].values[0]
    else:
        bg_sample = bg_df.sample(1)["feature"].values[0]

    if emotion in emo_groups.groups:
        emo_sample = emo_groups.get_group(emotion).sample(1)["features"].values[0]
    else:
        emo_sample = emotion_df.sample(1)["features"].values[0]

    fusion_rows.append({
        "text": row["call_text"],
        "label": row["emergency_type"],
        "urgency": row["urgency_level"],
        "bg_features": bg_sample,
        "emotion_features": emo_sample
    })

In [10]:
fusion_df = pd.DataFrame(fusion_rows)

In [11]:
fusion_df["bg_features"][0].shape


(40,)

In [12]:
fusion_df["emotion_features"][0].shape

(128, 200)

In [13]:
fusion_df.to_pickle("multimodal_emergency_dataset.pkl")

In [14]:
fusion_df.head(

)

,text,label,urgency,bg_features,emotion_features
0,oh my god a man is yelling and hitting someone...,domestic_violence,high,"[-253.202, 155.10077, -23.588371, 27.446623, 4...","[[-42.021408, -44.627045, -44.432507, -44.9077..."
1,listen flood water is entering houses and peop...,natural_disaster,critical,"[-149.53751, 95.927872, -80.926819, 5.1839571,...","[[-61.464966, -61.464966, -61.464966, -61.4649..."
2,can you help a bike hit a pedestrian and the r...,traffic_accident,high,"[-117.567825, 56.9537506, -73.7933273, 27.5287...","[[-72.85887, -72.85887, -72.85887, -72.85887, ..."
3,can you help a man just snatched a bag and ran...,robbery,high,"[-195.53075, 81.639801, 48.328751, 16.489645, ...","[[-47.776764, -50.512154, -51.719276, -45.7152..."
4,can you help a child fell into the swimming po...,drowning,critical,"[-500.00934, 75.786263, 9.3569269, 0.48533973,...","[[-70.58771, -70.58771, -70.58771, -70.58771, ..."
